# 02 - Document Chunking - How should we split documents for retrieval?

## Objective

Evaluate a document chunking strategy for the RAG pipeline.

The objective is to produce chunks that preserve semantic context while remaining suitable for embedding and retrieval.

This notebook focuses on experimentation rather than production implementation.

---

## Questions

- Why is chunking required?
- Which text splitter should be used?
- What chunk size is appropriate?
- Is chunk overlap required?
- Are the resulting chunks semantically coherent?

---

## Success Criteria

By the end of this notebook:

- Documents are successfully chunked.
- Chunk boundaries preserve context.
- Metadata is retained.
- Chunking parameters have been selected.

---

## Notes

The selected strategy will be implemented in the production codebase during the implementation phase.

In [4]:
from pathlib import Path
import re

from langchain_community.document_loaders import (
    Docx2txtLoader, PyMuPDFLoader, TextLoader,
)
from langchain_text_splitters import RecursiveCharacterTextSplitter

## Load documents
Reuse ingestion code of previous notebook. At this point I'm only experimenting, in production this would not be duplicated.

In [2]:
# project paths 
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw/knowledge_base"

LOADERS = {".pdf": PyMuPDFLoader, ".docx": Docx2txtLoader, ".md": TextLoader, ".txt": TextLoader}

documents = []
for path in sorted(RAW_DIR.rglob("*")):
    if path.is_dir():
        continue
    loader_cls = LOADERS.get(path.suffix.lower())
    if loader_cls is None:
        print(f"Skipping unsupported file: {path.name}")
        continue
    loader = loader_cls(str(path))
    documents.extend(loader.load())

print(f"Loaded {len(documents)} documents.")

Skipping unsupported file: .DS_Store
Skipping unsupported file: .DS_Store
Loaded 6 documents.


In [5]:
def clean_text(text):
    # Collapse single newlines (likely mid-sentence wraps) into spaces,
    # but preserve intentional paragraph breaks (blank lines).
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)
    # Collapse 3+ newlines down to a standard paragraph break.
    text = re.sub(r"\n{3,}", "\n\n", text)
    # Collapse repeated spaces/tabs.
    text = re.sub(r"[ \t]{2,}", " ", text)
    return text.strip()

for doc in documents:
    doc.page_content = clean_text(doc.page_content)

## Why chunking matters
A large language model has a limited context window.

Instead of embedding an entire document, the document is divided into smaller, overlapping chunks. This improves retrieval quality while preserving local context.

## Configure the text splitter
First parameters to work as baseline and to be evaluated

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
)

## Generate Chunks

In [7]:
chunks = text_splitter.split_documents(documents)

print(f"Generated {len(chunks)} chunks.")

Generated 50 chunks.


In [8]:
# Inspect chunk sizes 
chunk_lengths = [len(chunk.page_content) for chunk in chunks]

print(f"Minimum: {min(chunk_lengths)}")
print(f"Average: {sum(chunk_lengths) / len(chunk_lengths):.1f}")
print(f"Maximum: {max(chunk_lengths)}")

Minimum: 117
Average: 445.7
Maximum: 500


In [10]:
# Inspect chunk content
for i, chunk in enumerate(chunks[1:4]):
    print(f"Chunk {i+1}:")
    print(chunk.page_content)
    print("-" * 40)

Chunk 1:
are a strong fit for empowering the marketing team and different stakeholders through commercially relevant insights. I am a Principal Data Scientist with 5+ years of experience in consulting delivering advanced analytics projects in the industries of Financial Services and Retail. Further, I complement my technical experience with 4 additional years in finance and business analysis roles that have given me the basis for focusing on highly relevant, commercially sounding insights and modelling.
----------------------------------------
Chunk 2:
given me the basis for focusing on highly relevant, commercially sounding insights and modelling. I have advanced proficiency using technical tools like SQL, Python, and Agentic AI. However, my focus is on delivering actionable recommendations as the final goal of any advanced analytics initiative. My experience working with big 4 banks, neo banks, and some of the top retailers in Australia has shown me that a solid explainable model can

In [11]:
# inspect metadata of a chunk
chunks[1].metadata

{'producer': 'macOS Version 15.1 (Build 24B2082) Quartz PDFContext',
 'creator': '',
 'creationdate': "D:20260520123030Z00'00'",
 'source': '/Users/davidr/git_projects/ai-career-assistant/data/raw/knowledge_base/cover_letters/CoverLetter_DR_1.pdf',
 'file_path': '/Users/davidr/git_projects/ai-career-assistant/data/raw/knowledge_base/cover_letters/CoverLetter_DR_1.pdf',
 'total_pages': 1,
 'format': 'PDF 1.4',
 'title': '',
 'author': '',
 'subject': '',
 'keywords': '',
 'moddate': "D:20260520123030Z00'00'",
 'trapped': '',
 'modDate': "D:20260520123030Z00'00'",
 'creationDate': "D:20260520123030Z00'00'",
 'page': 0}

## Experimenting with parameters

In [19]:
configs = [
    {"chunk_size": 500, "chunk_overlap": 100},
    {"chunk_size": 1000, "chunk_overlap": 200},
]

for config in configs:
    print("= " * 50)
    splitter = RecursiveCharacterTextSplitter(**config)
    test_chunks = splitter.split_documents(documents)

    print(config)
    print(f"Chunks: {len(test_chunks)}")
    print("=" * 50)

    # Inspect chunk content
    for i, chunk in enumerate(test_chunks[1:4]):
        print(f"Chunk {i+1}:")
        print(chunk.page_content)
        print("-" * 80)
    print("= " * 50)

= = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = 
{'chunk_size': 500, 'chunk_overlap': 100}
Chunks: 50
Chunk 1:
are a strong fit for empowering the marketing team and different stakeholders through commercially relevant insights. I am a Principal Data Scientist with 5+ years of experience in consulting delivering advanced analytics projects in the industries of Financial Services and Retail. Further, I complement my technical experience with 4 additional years in finance and business analysis roles that have given me the basis for focusing on highly relevant, commercially sounding insights and modelling.
--------------------------------------------------------------------------------
Chunk 2:
given me the basis for focusing on highly relevant, commercially sounding insights and modelling. I have advanced proficiency using technical tools like SQL, Python, and Agentic AI. However, my focus is on delivering actionable recommendations as 

## Final configuration selected

In [ ]:
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

## Conclusion

### Decision

Use `RecursiveCharacterTextSplitter` with:

- Chunk size: **500**
- Chunk overlap: **100**

### Rationale

- Preserves local context.
- Produces consistent chunk sizes.
- Retains document metadata.
- Provides a strong baseline for semantic retrieval.

### Next Step

Generate embeddings for the resulting chunks.